In [11]:
import json
import pandas as pd
import numpy as np
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

In [10]:
LOG_DIR = "./logs"
TASK_NAME = "listing_generator_eval"

RUN_PIPELINE = True 

In [15]:
"""
Run Full Pipeline
"""
from src.property_listing.eval.pipeline import run_eval

evaluation_results = run_eval()

print(evaluation_results[0].location)




ModuleNotFoundError: No module named 'property_listing'

In [7]:

from inspect_ai.log import read_eval_log

log = read_eval_log(LOG_PATH)
print(log)

version=2 status='success' eval=EvalSpec(eval_set_id=None, eval_id='7jHYSRKsYdeGBhE2eKhpxp', run_id='jCdyHm5yPE3pAjbUL4pA4a', created='2026-06-15T13:48:18+00:00', task='listing_generator_eval', task_id='kVS3m5AS6wxU9VpHB5p9bv', task_version=0, task_file='src/property_listing/eval/eval_pipeline.py', task_display_name='listing_generator_eval', task_registry_name='listing_generator_eval', task_attribs={}, task_args={}, task_args_passed={}, solver=None, solver_args=None, solver_args_passed={}, tags=None, dataset=EvalDataset(name=None, location=None, samples=2, sample_ids=['10482', '10483'], shuffled=False), sandbox=None, model='none/none', model_generate_config=GenerateConfig(max_retries=None, timeout=None, attempt_timeout=None, max_connections=None, adaptive_connections=None, system_message=None, max_tokens=None, top_p=None, temperature=None, stop_seqs=None, best_of=None, frequency_penalty=None, presence_penalty=None, logit_bias=None, seed=None, top_k=None, num_choices=None, logprobs=None

In [8]:
import pandas as pd

rows = []
for s in log.samples:
    rows.append({
        "id": s.id,
        "score": getattr(s.score, "value", None),
        "explanation": getattr(s.score, "explanation", None),
        "output": getattr(s.output, "text", None),
        "metadata": getattr(s.score, "metadata", None),
    })

df = pd.DataFrame(rows)
df.head()

The 'score' field is deprecated. Access sample scores through 'scores' instead.


,id,score,explanation,output,metadata
0,10482,0.7000,Groundedness Score: 0.70 (14/20 claims verifie...,None,{'audit_checks': [{'claim': 'The Luminary Pent...
1,10483,0.9375,Groundedness Score: 0.94 (15/16 claims verifie...,None,{'audit_checks': [{'claim': 'Property name is ...


In [ ]:
def expand_metadata(df):
    meta = df["metadata"].apply(lambda x: x if isinstance(x, dict) else {})
    meta_df = pd.json_normalize(meta)
    return pd.concat([df.drop(columns=["metadata"]), meta_df], axis=1)

df = expand_metadata(df)
df.head()

In [ ]:
summary = {
    "num_samples": len(df),
    "mean_score": df["score"].mean(),
    "min_score": df["score"].min(),
    "max_score": df["score"].max(),
}

summary

In [ ]:
plt.figure()
df["score"].hist(bins=10)
plt.title("Score Distribution")
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.show()

In [ ]:
low = df[df["score"] < df["score"].quantile(0.25)]
low[["sample_id", "score", "explanation"]].head(10)

In [ ]:
if "total_claims" in df.columns:
    df["grounded_ratio"] = df["score"]
    
    plt.figure()
    df["grounded_ratio"].hist(bins=10)
    plt.title("Groundedness Ratio Distribution")
    plt.xlabel("Grounded Ratio")
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
def extract_failed_claims(row):
    meta = row.get("metadata", {})
    if isinstance(meta, dict):
        checks = meta.get("audit_checks", [])
        return [c for c in checks if not c.get("verdict")]
    return []

df["failed_claims"] = df.apply(extract_failed_claims, axis=1)

In [ ]:
from collections import Counter

all_failed = []
for item in df["failed_claims"]:
    for c in item:
        all_failed.append(c.get("claim"))

Counter(all_failed).most_common(20)

In [ ]:
def view_sample(sample_id):
    row = df[df["sample_id"] == sample_id].iloc[0]
    
    print("SAMPLE ID:", sample_id)
    print("\nSCORE:", row["score"])
    print("\nEXPLANATION:\n", row["explanation"])
    print("\nOUTPUT:\n", row.get("output"))
    
# Example:
view_sample(df["sample_id"].iloc[0])